1. construct the DCOPF latex tutorial
2. check the PU in the formulation, since the 30bus is infeasible

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from scipy.stats import multivariate_normal
import numpy as np
from tqdm import tqdm
file=".\\old\\DCOPF-main\\DCOPF-main\\excel_outputs\\pglib_opf_case1951_rte.xlsx"
mpc_data = pd.read_excel(file, sheet_name=['baseMVA', 'bus', 'gen', 'gencost', 'branch'])

In [2]:
def omega_sample(Pd, sigma_scaling=0.03, nsamples=1000):
    stdomega = [sigma_scaling*Pd[b] for b in buses]
    nonzeroindices = [i for i in range(len(stdomega)) if stdomega[i] > 1e-5]
    mean = np.zeros(len(nonzeroindices))
    cov = np.diag(list(map(stdomega.__getitem__, nonzeroindices)))**2
    omega = multivariate_normal.rvs(mean=mean, cov=cov, size=nsamples)
    omega_samples = np.zeros((len(buses), nsamples))
    omega_samples[nonzeroindices] = omega.T if omega.ndim == 2 else omega[:, np.newaxis]
    return omega_samples

In [3]:
# Create gurobipy Model
model = gp.Model("DCOPF")
# === Sets ===
buses = mpc_data['bus']['bus_i'].tolist()
buses_index_busID = dict(zip(mpc_data['bus'].index,mpc_data['bus']['bus_i']))
buses_busID_index = dict(zip(mpc_data['bus']['bus_i'],mpc_data['bus'].index))
gens = mpc_data['gen']['gen_ID'].tolist()
branches = mpc_data['branch'].index.tolist()
branches_ftbus = dict(zip(branches,mpc_data['branch'][['bus_i', 'bus_j']].values))
# === Parameters ===
# Generator cost coefficients (all costs are incorporated)
c = {}
for i in mpc_data['gencost']["gen_ID"]:
    c[i] = [mpc_data['gencost']['c2'][i-1],mpc_data['gencost']['c1'][i-1],mpc_data['gencost']['c0'][i-1]]
# Bus power demand (MW)
Pd = dict(zip(mpc_data['bus']['bus_i'], mpc_data['bus']['Pd']))
# shunt conductance (MW demanded at V = 1.0 p.u.)
Gs = dict(zip(mpc_data['bus']['bus_i'], mpc_data['bus']['Gs']))
# Generator capacity limits (MW)
Pmax = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['Pmax']))    
Pmin = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['Pmin']))
# Transmission line limits (MW)
Pmax_line = dict(zip(branches, mpc_data['branch']['rateA']))
Pmin_line = dict(zip(branches, -mpc_data['branch']['rateA']))
# Line susceptance (1/X), assuming per unit values
B = dict(zip(branches, mpc_data['branch']['x']/(mpc_data['branch']['x']**2+mpc_data['branch']['r']**2)))
# Generator bus assignment
gen_bus = dict(zip(mpc_data['gen']['gen_ID'], mpc_data['gen']['bus_i']))
# === Variables ===
Pg = model.addVars(gens, lb=Pmin, ub=Pmax, vtype=GRB.CONTINUOUS, name="Pg")
theta = model.addVars(buses, lb=-100, ub=100, vtype=GRB.CONTINUOUS, name="theta")
P_flow = model.addVars(branches, lb=Pmin_line, ub=Pmax_line, vtype=GRB.CONTINUOUS, name="P_flow")
omega = model.addVars(range(len(buses)), lb=-GRB.INFINITY, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name="omega")
# === Objective Function (Minimize Generation Cost, all costs are incorporated) ===
model.setObjective(gp.quicksum(c[i][0]*Pg[i] + c[i][1]*Pg[i] + c[i][2] for i in gens), GRB.MINIMIZE)
# === Power Balance Constraints ===
for b in buses:
    expr = (gp.quicksum(Pg[i] for i in gen_bus if gen_bus[i] == b)
            + gp.quicksum(P_flow[l] for l, ft in branches_ftbus.items() if ft[1] == b)
            - gp.quicksum(P_flow[l] for l, ft in branches_ftbus.items() if ft[0] == b))
    model.addConstr(expr == Pd[b] + Gs[b] - omega[buses_busID_index[b]] , name=f"power_balance_{b}")
# === Line Flow Constraints (DC Power Flow) ===
for l in branches:
    model.addConstr(P_flow[l] == B[l]*(theta[branches_ftbus[l][0]] - theta[branches_ftbus[l][1]]), name=f"line_flow_{l}")
# === Reference Bus Constraint (Slack Bus) ===
ref_bus_index = mpc_data['bus'][mpc_data['bus']['type'] == 3].index[0]
model.addConstr(theta[buses_index_busID[ref_bus_index]] == 0, name="theta_ref") # buses_index_busID[ref_bus_index] gets the ref bus
# === Solve Model Using Gurobi ===
model.Params.OptimalityTol = 1e-8 # Higher precision
model.setParam('OutputFlag', 0) # suppress the output
# model.optimize() 

Set parameter OptimalityTol to value 1e-08


In [4]:
class OPF_Scenarios:
    def __init__(self,noptimal, scenarios, solutions, cbases, rbases, whichbasis, whichscenario):
        self.noptimal = noptimal
        self.scenarios = scenarios
        self.solutions = solutions
        self.cbases = cbases
        self.rbases = rbases
        self.whichbasis = whichbasis
        self.whichscenario = whichscenario
        
def OPFScenarios(model, omega, omega_samples):
    nsamples = omega_samples.shape[1]
    status = [None] * nsamples
    soln_p = np.zeros((nsamples, len(gens)))
    cbases = {}
    rbases = {}
    noptimal = 0
    for s in tqdm(range(nsamples)):
        model.setAttr("LB", omega, omega_samples[:,s])
        model.setAttr("UB", omega, omega_samples[:,s])
        # for num in omega:
        #     omega[num].lb = omega_samples[num][s]
        #     omega[num].ub = omega_samples[num][s]
#         m.model.setParam('OutputFlag', 0) # suppress the output
        model.optimize()
        status[s] = model.status
        if status[s] == GRB.OPTIMAL:
            soln_p[s,:] = list(model.getAttr('x', Pg).values())
            noptimal += 1
            cbasis = tuple(model.getAttr('Vbasis', model.getVars()))
            rbasis = tuple(model.getAttr('Cbasis', model.getConstrs()))
            cbases[cbasis] = cbases.get(cbasis, [])
            rbases[rbasis] = rbases.get(rbasis, [])
            cbases[cbasis].append(noptimal)
            rbases[rbasis].append(noptimal)
    assert noptimal == sum(1 for stat in status if stat == 2), 'Mismatch in optimal scenario count'
    sample_p = soln_p[np.array(status)==GRB.OPTIMAL,:]
    sample_omega = omega_samples[:, np.array(status)==GRB.OPTIMAL]
    colbases = list(cbases.keys())
    rowbases = list(rbases.keys())
    whichcol = dict(zip(colbases, range(len(colbases))))
    whichrow = dict(zip(rowbases, range(len(rowbases))))
    whichbasis = np.zeros((noptimal, 2), dtype=int)
    for ckey in cbases.keys():
        whichbasis[cbases.get(ckey)[-1]-1,0] = whichcol[ckey]
    for rkey in rbases.keys():
        whichbasis[rbases.get(rkey)[-1]-1,1] = whichrow[rkey]
    whichscenario = {}
    for i in range(noptimal):
        basiskey = (whichbasis[i,0], whichbasis[i,1])
        whichscenario[basiskey] = whichscenario.get(basiskey, [])
        whichscenario[basiskey].append(i)
    return OPF_Scenarios(noptimal, sample_omega, sample_p, colbases, rowbases, whichbasis, whichscenario)

In [ ]:
# omega_samples = omega_sample(Pd, sigma_scaling=0.03, nsamples=5000)
# scenarios = OPFScenarios(model, omega, omega_samples)

100%|██████████| 5000/5000 [00:51<00:00, 96.89it/s] 


5000

In [ ]:
# def __init__(self,noptimal, scenarios, solutions, cbases, rbases, whichbasis, whichscenario):
# scenarios.whichscenario

{(0, 0): [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
  156,
  1